# **GroupDNA - MINOR PROJECT 1**

# Hostel Bois 4ever Analysis

---


**Name:** Piyush Sable
**Batch:** Data Science
**Date:** 21-09-2026

Analysing whatsApp group chats usinf Python and Numpy

##**Feature 1: The Chat Parser**

In [1]:
with open('/content/hostel_bois_Dataset.txt', 'r', encoding='utf-8') as f:
    lines = f.read().split('\n')
print(len(lines))


3178


In [24]:
print("="*50)
print("FEATURE 1: THE CHAT PARSER")
print("="*50)

messages = []
system_count = 0
media_count = 0
deleted_count = 0

for line in lines:
    line = line.strip()
    if line == '':
        continue
    # Split timestamp from the rest of the message
    parts = line.split(' - ', 1)
    if len(parts) == 1:
        # This is a continuation of the previous message
        if messages:
            messages[-1]['text'] = messages[-1]['text'] + ' ' + line
        continue
    # Get timestamp
    timestamp = parts[0]

    # Get sender and message text
    rest = parts[1]
    pieces = rest.split(': ', 1)

    if len(pieces) == 1:
        # System message
        system_count += 1
        continue
    sender = pieces[0]
    text = pieces[1]

    # Handle media/deleted messages
    if text == '<Media omitted>':
        media_count += 1
    elif text == '<Deleted>':
        deleted_count += 1
    else:
        message = {
            'timestamp': timestamp,
            'sender': sender,
            'text': text
        }
        messages.append(message)

# Count participants
total_participants = len(set(m['sender'] for m in messages))

print(f"Successfully parsed {len(messages)} messages from {total_participants} participants")
print(f"Skipped {system_count} system messages, {media_count} media-omitted, {deleted_count} deleted messages")


FEATURE 1: THE CHAT PARSER
Successfully parsed 3142 messages from 6 participants
Skipped 4 system messages, 32 media-omitted, 0 deleted messages


##**Feature 2: Group Overview**


In [8]:

from datetime import datetime

counts = {}
for m in messages:
    name = m['sender']
    counts[name] = counts.get(name, 0) + 1

ranked = sorted(counts.items(), key=lambda pair: pair[1], reverse=True)
total = len(messages)

print("GROUP OVERVIEW")
print("-" * 40)
print("Total messages:", total)
print("Participants  :", len(counts))
print()

for name, count in ranked:
    percent = count / total * 100
    print(f"{name:<8} {count:>4}  ({percent:.1f}%)")


GROUP OVERVIEW
----------------------------------------
Total messages: 3142
Participants  : 6

Rahul     946  (30.1%)
Priya     714  (22.7%)
Neha      627  (20.0%)
Aman      486  (15.5%)
Karan     347  (11.0%)
Vikas      22  (0.7%)


##**Feature 3: Moat Active Day and Hour**

In [23]:
print("="*50)
print("FEATURE 3: BUSIEST DAY AND HOUR")
print("="*50)

days = {}
hours = {}
for msg in messages:

    date_time = datetime.strptime(
        msg['timestamp'],
        '%d/%m/%y, %H:%M'
    )
    day = date_time.date()
    hour = date_time.hour

    if day not in days:
        days[day] = 0

    days[day] += 1

    if hour not in hours:
        hours[hour] = 0

    hours[hour] += 1

max_day = max(days, key=days.get)
max_hour = max(hours, key=hours.get)

print(
    f"Busiest day: {max_day.strftime('%d %B %Y')} "
    f"({days[max_day]} messages)"
)
print(
    f"Busiest hour: {max_hour:02d}:00 - "
    f"{(max_hour + 1) % 24:02d}:00 "
    f"({hours[max_hour]} messages)"
)
print()

FEATURE 3: BUSIEST DAY AND HOUR
Busiest day: 04 May 2024 (74 messages)
Busiest hour: 18:00 - 19:00 (246 messages)



##**Feature 4: Activity HeatMap (Numpy)**

In [22]:
import numpy as np
from datetime import datetime

# Get people
people = sorted(set(m['sender'] for m in messages))

# Map person name -> row number
person_row = {
    name: i
    for i, name in enumerate(people)
}

# IMPORTANT:
# 24 columns = one column for EACH hour of the day
heatmap = np.zeros((len(people), 24), dtype=int)

for m in messages:

    dt = datetime.strptime(
        m['timestamp'],
        '%d/%m/%y, %H:%M'
    )
    heatmap[person_row[m['sender']]][dt.hour] += 1


def shade_block(value, row_max):

    if row_max == 0:
        return '. '

    ratio = value / row_max
    if ratio <= 0.25:
        return '. '
    elif ratio <= 0.50:
        return '░ '
    elif ratio <= 0.75:
        return '▒ '
    else:
        return '█ '


print()
print("\n" + "="*50 + "\n")
print(" ACTIVITY HEATMAP (messages by hour, shown every 3 hours)")
print("\n" + "="*50 + "\n")
print("           00  03  06  09  12  15  18  21")


for name in people:
    row = heatmap[person_row[name]]
    # Maximum for THIS PERSON
    row_max = row.max()
    line = f" {name:<8}"

    for hour in range(0, 24, 3):

        line += " " + shade_block(
            row[hour],
            row_max
        )

    print(line)






 ACTIVITY HEATMAP (messages by hour, shown every 3 hours)


           00  03  06  09  12  15  18  21
 Aman     ▒  ▒  .  .  .  .  .  . 
 Karan    .  .  .  ░  █  ▒  ▒  ░ 
 Neha     .  .  .  █  ▒  .  █  ░ 
 Priya    .  .  .  █  █  ░  ▒  ░ 
 Rahul    .  .  .  .  ▒  ▒  █  █ 
 Vikas    .  .  .  ░  ░  ░  ▒  ░ 


##**Feature 5: Top Word**

In [19]:

stop_words = {
    'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on',
    'for', 'you', 'it', 'my', 'me', 'we', 'are', 'am', 'was',
    'this', 'that', 'so', 'at', 'he', 'his', 'have', 'how',
    'about', 'today', 'not', 'be', 'with', 'do', 'did', 'if',
    'but', 'all', 'just', 'your', 'has', 'will', 'can', 'what',
    'when', 'why', 'who', 'they', 'them', 'their', 'which',
    'everyone', 'telling', 'from', 'up', 'one', 'had', 'been',
    'were', 'as', 'by', 'an', 'us', 'our', 'then', 'than',
    'some', 'more', 'out', 'no', 'yes', 'get', 'got', 'going',
    'go', 'would', 'could', 'should', 'there', 'here', 'only',
    'also', 'into', 'over', 'again', 'still', 'even', 'now'
}


punctuation = '.,!?"\'()'


word_counts = {}


for message in messages:

    text = message['text']


    if text == '<Media omitted>' or text == 'This message was deleted':
        continue

    # Convert the message to lowercase
    text = text.lower()

    # Split the message into words
    words = text.split()

    for word in words:

        word = word.strip(punctuation)

        if word == '' or word in stop_words:
            continue

        # Count the word
        if word in word_counts:
            word_counts[word] += 1
        else:
            word_counts[word] = 1


sorted_words = sorted(
    word_counts.items(),
    key=lambda x: x[1],
    reverse=True
)

# Takinng the first 10 words
top_words = sorted_words[:10]


# Print the result
print("\n" + "="*50 + "\n")
print(" THIS GROUP'S FAVOURITE WORDS")
print("\n" + "="*50 + "\n")

# Find the highest count
highest_count = top_words[0][1]

for word, count in top_words:

    # Calculate the length of the bar
    bar_length = int((count / highest_count) * 20)

    # Creating the bar
    bar = '█' * bar_length

    print(f" {word:<10}{bar} {count}")
    print()





 THIS GROUP'S FAVOURITE WORDS


 guys      ████████████████████ 318

 hai       ████████████████ 268

 bhai      ██████████ 160

 started   █████████ 150

 scene     █████████ 145

 entire    █████████ 145

 please    ████████ 141

 anyone    ████████ 139

 yaar      ████████ 139

 kya       ████████ 133



##**Feature 6: Response speed & Silent streak**

In [20]:
from datetime import datetime, timedelta

# --- (a) Average response time per person ---

response_times = {}

for person in people:
    response_times[person] = []
previous_sender = None
previous_time = None

for message in messages:
    current_time = datetime.strptime(
        message['timestamp'],
        '%d/%m/%y, %H:%M'
    )
    current_sender = message['sender']
    if previous_sender is not None and current_sender != previous_sender:
        gap = current_time - previous_time
        response_times[current_sender].append(gap.total_seconds())
    previous_sender = current_sender
    previous_time = current_time


average_time = {}
for person in people:
    times = response_times[person]
    if len(times) > 0:
        average_time[person] = sum(times) / len(times)
    else:
        average_time[person] = 0


fastest_person = people[0]
slowest_person = people[0]

for person in people:
    if average_time[person] < average_time[fastest_person]:
        fastest_person = person
    if average_time[person] > average_time[slowest_person]:
        slowest_person = person

print("\n" + "="*50 + "\n")
print("RESPONSE PATTERNS")
print("\n" + "="*50 + "\n")

print(
    f"Fastest replier: {fastest_person} "
    f"({average_time[fastest_person] / 60:.1f} minutes)"
)
print(
    f"Slowest replier: {slowest_person} "
    f"({average_time[slowest_person] / 3600:.1f} hours)"
)


# --- (b) Longest silent streak ---

first_dt = datetime.strptime(
    messages[0]['timestamp'],
    '%d/%m/%y, %H:%M'
)
last_dt = datetime.strptime(
    messages[-1]['timestamp'],
    '%d/%m/%y, %H:%M'
)
total_days = (last_dt.date() - first_dt.date()).days + 1

active_days = {}

for person in people:
    active_days[person] = set()
for message in messages:
    date = datetime.strptime(
        message['timestamp'],
        '%d/%m/%y, %H:%M'
    ).date()
    person = message['sender']
    active_days[person].add(date)


silent_streaks = {}
for person in people:
    longest = 0
    current = 0
    for i in range(total_days):
        day = first_dt.date() + timedelta(days=i)
        if day not in active_days[person]:
            current += 1
            if current > longest:
                longest = current
        else:
            current = 0
    silent_streaks[person] = longest

print()
print("LONGEST SILENT STREAKS")

for person in people:
    print(f"{person}: {silent_streaks[person]} days")




RESPONSE PATTERNS


Fastest replier: Vikas (34.9 minutes)
Slowest replier: Aman (0.9 hours)

LONGEST SILENT STREAKS
Aman: 0 days
Karan: 0 days
Neha: 0 days
Priya: 0 days
Rahul: 0 days
Vikas: 11 days


##**Feature 7: Personlity Architype detection**

In [21]:

# group messages by person
person_messages = {}
for person in people:
    person_messages[person] = []

for message in messages:
    person_messages[message['sender']].append(message)


# find how many days each person stayed silent (as a %)
silent_days_percent = {}
for person in people:
    active_days = set()
    for message in person_messages[person]:
        date = datetime.strptime(message['timestamp'], '%d/%m/%y, %H:%M').date()
        active_days.add(date)

    silent_days = total_days - len(active_days)
    silent_days_percent[person] = silent_days / total_days * 100


# words used to detect caring / funny messages
caring_words = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water']
laugh_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']

scores = {}
for person in people:
    messages_list = person_messages[person]
    mom = 0
    night = 0
    storyteller = 0
    drama = 0
    comedian = 0
    questions = 0

    for message in messages_list:
        text = message['text'].lower()

        # checks for caring words
        for word in caring_words:
            if word in text:
                mom += 1

        # checks message hour
        hour = datetime.strptime(message['timestamp'], '%d/%m/%y, %H:%M').hour
        if hour == 23 or hour <= 4:
            night += 1

        # counting total words for now, will average later
        storyteller += len(message['text'].split())

        # check if all caps or multiple exclamation marks
        letters = ''.join(c for c in message['text'] if c.isalpha())
        if len(letters) >= 3 and letters.isupper():
            drama += 1
        elif message['text'].count('!') >= 2:
            drama += 1

        # checks for laugh words
        for word in laugh_words:
            if word in text:
                comedian += 1
                break

        # does message ends with ?
        if message['text'].strip().endswith('?'):
            questions += 1

    # turn counts into averages /percentages
    total_messages = len(messages_list)

    if total_messages > 0:
        storyteller = storyteller / total_messages
        night = night / total_messages * 100
        drama = drama / total_messages * 100
        comedian = comedian / total_messages * 100
        questions = questions / total_messages * 100
    else:
        storyteller = 0
        night = 0
        drama = 0
        comedian = 0
        questions = 0

    # spammer - longest streak of consecutive messages by this person
    longest_burst = 0
    current_burst = 0
    previous_sender = None

    for message in messages:
        if message['sender'] == person:
            if previous_sender == person:
                current_burst += 1
            else:
                current_burst = 1
            if current_burst > longest_burst:
                longest_burst = current_burst
        else:
            current_burst = 0
        previous_sender = message['sender']
    spammer = longest_burst
    scores[person] = {
        'THE SPAMMER': spammer,
        'THE GROUP MOM': mom,
        'THE NIGHT OWL': night,
        'THE STORYTELLER': storyteller,
        'THE DRAMA QUEEN': drama,
        'THE GHOST': silent_days_percent[person],
        'THE COMEDIAN': comedian,
        'THE QUESTION MASTER': questions
    }


# minimum score needed to qualify for each archetype
thresholds = {
    'THE SPAMMER': 3,
    'THE GROUP MOM': 1,
    'THE NIGHT OWL': 60,
    'THE STORYTELLER': 30,
    'THE DRAMA QUEEN': 30,
    'THE GHOST': 60,
    'THE COMEDIAN': 1,
    'THE QUESTION MASTER': 25
}

# order matters
order = [
    'THE SPAMMER',
    'THE GROUP MOM',
    'THE NIGHT OWL',
    'THE STORYTELLER',
    'THE DRAMA QUEEN',
    'THE GHOST',
    'THE COMEDIAN',
    'THE QUESTION MASTER'
]
results = {}

for archetype in order:
    best_person = None
    best_score = 0
    for person in people:
        if person in results:
            continue
        score = scores[person][archetype]
        if score >= thresholds[archetype] and score > best_score:
            best_person = person
            best_score = score
    if best_person is not None:
        results[best_person] = (archetype, best_score)

print("\n" + "="*50 + "\n")
print("PERSONALITY ARCHETYPES")
print("\n" + "="*50 + "\n")

for person in people:
    if person in results:
        archetype, score = results[person]
        print(f"{person}: {archetype} (score {score:.1f})")
    else:
        print(f"{person}: no archetype")



PERSONALITY ARCHETYPES


Aman: THE NIGHT OWL (score 80.5)
Karan: THE STORYTELLER (score 56.7)
Neha: THE DRAMA QUEEN (score 63.0)
Priya: THE GROUP MOM (score 605.0)
Rahul: THE SPAMMER (score 13.0)
Vikas: THE GHOST (score 73.3)
